In [ ]:
from pathlib import Path
import pandas as pd
from glob import glob
from tqdm import tqdm
import os

In [ ]:
src_path = '../../data/stock_data/raw/'
dst_path = '../../data/stock_data/processed/'
ai_ready_path = '../../data/stock_data/ai_ready/'




In [ ]:
# 모델 입력에 필요한 원본 컬럼과 파생 계산용 컬럼
source_columns = [
    'symbol',
    'timestamp_kst',
    'open',
    'high',
    'low',
    'close',
    'volume',
    'trade_strength',
    'aggressive_buy_volume',
    'aggressive_sell_volume',
    'session_vwap',
    'return_pct',
    'bid_ask_imbalance',
    'avg_bid_size',
    'avg_ask_size',
    'avg_spread_pct',
    'ema9',
    'ema20',
    'trade_notional_from_ticks',
]

output_columns = [
    'symbol',
    'timestamp_kst',
    'open',
    'high',
    'low',
    'close',
    'volume',
    'trade_strength',
    'aggressive_buy_volume',
    'aggressive_sell_volume',
    'session_vwap',
    'return_pct',
    'bid_ask_imbalance',
    'avg_bid_size',
    'avg_ask_size',
    'avg_spread_pct',
    'ema9',
    'ema20',
    'volume_ratio_ma20',
    'vwap_gap_ratio',
    'ema_momentum',
    'aggressive_buy_ratio',
    'capital_inflow_ratio_ma20',
    'spread_ratio_ma20',
    'orderbook_imbalance_strength',
]


def safe_divide(numerator, denominator):
    # 0으로 나눈 값은 inf 대신 NaN으로 남긴다.
    return numerator.div(denominator.where(denominator.ne(0)))
src_files_list = glob(src_path + '**/*_enriched.csv')

if not src_files_list:
    raise FileNotFoundError(f'enriched CSV를 찾지 못했습니다: {src_path}')

dst_root = Path(dst_path)
dst_root.mkdir(parents=True, exist_ok=True)

# processed 폴더는 평면 구조로 유지한다. 원본 파일의 날짜와 run ID를 파일명에 보존한다.
source_paths = [Path(file_path) for file_path in sorted(src_files_list)]
expected_output_names = {
    source_file.name.removesuffix('_enriched.csv') + '_features.csv'
    for source_file in source_paths
}

# 이전 실행에서 만든 종목 통합 파일 또는 사라진 원본의 stale feature 파일을 제거한다.
for existing_path in dst_root.glob('*_features.csv'):
    if existing_path.name not in expected_output_names:
        existing_path.unlink()

save_records = []
processed_files_list = []
warmup_bars = 20
total_input_rows = 0
total_saved_rows = 0

for source_file in tqdm(source_paths, desc='Build per-file features'):
    frame = pd.read_csv(source_file, usecols=source_columns, low_memory=False)
    frame['timestamp_kst'] = pd.to_datetime(frame['timestamp_kst'], errors='raise')
    frame = frame.sort_values('timestamp_kst').reset_index(drop=True)

    if frame['symbol'].nunique() != 1:
        raise ValueError(f'한 파일에 여러 종목이 있습니다: {source_file}')
    if frame.duplicated(['symbol', 'timestamp_kst']).any():
        raise ValueError(f'동일 symbol/timestamp_kst 중복이 있습니다: {source_file}')

    # rolling은 다른 날짜 파일과 연결하지 않고 현재 종목·거래일 파일 내부에서만 계산한다.
    volume_mean20 = frame['volume'].rolling(20, min_periods=20).mean()
    notional_mean20 = frame['trade_notional_from_ticks'].rolling(20, min_periods=20).mean()
    spread_mean20 = frame['avg_spread_pct'].rolling(20, min_periods=5).mean()

    frame['volume_ratio_ma20'] = safe_divide(frame['volume'], volume_mean20)
    frame['vwap_gap_ratio'] = safe_divide(
        frame['close'] - frame['session_vwap'],
        frame['session_vwap'],
    )
    frame['ema_momentum'] = frame['ema9'] - frame['ema20']
    frame['aggressive_buy_ratio'] = safe_divide(
        frame['aggressive_buy_volume'],
        frame['volume'],
    )
    frame['capital_inflow_ratio_ma20'] = safe_divide(
        frame['trade_notional_from_ticks'],
        notional_mean20,
    )
    frame['spread_ratio_ma20'] = safe_divide(
        frame['avg_spread_pct'],
        spread_mean20,
    )
    frame['orderbook_imbalance_strength'] = (
        frame['bid_ask_imbalance'] - 0.5
    ).abs()

    # rolling 지표 준비 구간인 파일별 최초 20개 봉은 최종 데이터에서 제거한다.
    result = frame.iloc[warmup_bars:][output_columns].reset_index(drop=True).copy()

    # 정상 범위가 0 이상인 결측 컬럼은 학습용 sentinel -1로 저장한다.
    missing_value_columns = [
        'trade_strength',
        'avg_spread_pct',
        'avg_bid_size',
        'avg_ask_size',
        'bid_ask_imbalance',
        'spread_ratio_ma20',
        'orderbook_imbalance_strength',
    ]
    result[missing_value_columns] = result[missing_value_columns].fillna(-1.0)
    output_name = source_file.name.removesuffix('_enriched.csv') + '_features.csv'
    output_path = dst_root / output_name
    result.to_csv(output_path, index=False)

    symbol = str(result['symbol'].iloc[0])
    total_input_rows += len(frame)
    total_saved_rows += len(result)
    processed_files_list.append(str(output_path))
    save_records.append({
        'symbol': symbol,
        'rows': len(result),
        'start': result['timestamp_kst'].min(),
        'end': result['timestamp_kst'].max(),
        'source_file': source_file.name,
        'output_file': output_path.name,
    })

save_summary = pd.DataFrame(save_records)
print(f'입력 파일: {len(source_paths):,}개')
print(f'원본 전체 행: {total_input_rows:,}개')
print(f'제거한 초기 봉: {total_input_rows - total_saved_rows:,}개')
print(f'저장 전체 행: {total_saved_rows:,}개')
print(f'고유 종목: {save_summary["symbol"].nunique():,}개')
print(f'저장 파일: {len(save_summary):,}개')
print(f'저장 위치: {dst_root.resolve()}')
display(save_summary.head(20))


In [ ]:
dst_files_list = sorted(glob(str(dst_root / '*_features.csv')))
pd.read_csv(dst_files_list[0])